# Estudio 06 · Fase 3 · Inicios de brote (3a1, 3a2 y 3b)

Preregistro en `code/preregistration.py → PHASE3` (commit `6574669`, aprobado por Rosa con dos ajustes antes de
entrenar). Base: L (C3 · E3 · S4_s3 desde 2005, Platt por horizonte con recorte 1e-6).

- **3a1:** peso 5 a los orígenes con algún objetivo en brote nuevo. **3a2:** peso por temporada mediana(C)/C_s
  recortado a [0,25, 1]. **3b:** dos regímenes por el estado en la semana de origen.
- Cada una contra L con la regla de 5 temporadas; se adopta con «gana» en ≥ 2 de 3 horizontes y sin «pierde» ni
  «compensa». 2010 aparte; 2024 contraste.
- **3c** (B1 + B2 en el modelo de inicios) no se corre aquí: Rosa revisa antes B1 y B2 sobre la serie larga con G10.


In [1]:
import json
import sys

import pandas as pd

from denguekay.config.paths import RESEARCH
from denguekay.evaluation import episodes as diag
from denguekay.model import tracking
from denguekay.model.device import xgboost_settings

STUDY = RESEARCH / "06_model_improvement"
sys.path.insert(0, str(STUDY / "code"))
import phase2_calibration as p2
import phase3_onsets as p3
import preregistration as pr

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 300)
METRICS = STUDY / "docs" / "metricas" / "fase3.json"
device = xgboost_settings()
inp = p3.phase3_inputs()
device


{'device': 'cuda', 'n_jobs': 12}

## 1 · Guarda G8 (G9 es un test)


In [2]:
saved = p2.read_l_predictions()
g8 = p3.guard_g8(inp, saved[17])
if not g8["pasa"]:
    raise RuntimeError(f"Falla G8: se detiene la fase. {g8}")
g8


{'sin_pesos': {'filas': 183105,
  'filas_guardadas': 183105,
  'cruzadas': 183105,
  'diferencia_maxima': {'casos_predichos': 0.0, 'probabilidad_brote': 0.0},
  'pasa': True},
 'pesos_1': {'diferencia_maxima': {'casos_predichos': 0.0,
   'probabilidad_brote': 0.0}},
 'pasa': True}

## 2 · Referencia L y variantes (5 semillas)


In [3]:
L = p3.reference_l(inp["serie"])
V = {v: p3.run_variant(inp, v, pr.SEEDS) for v in ("3a1", "3a2", "3b")}
p3.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for v, res in V.items():
    pd.concat(res["predicciones"].values(), ignore_index=True).to_csv(
        p3.OUTPUT_DIR / f"predicciones_{v}.csv.gz", index=False)
    res["ajustes"].to_csv(p3.OUTPUT_DIR / f"ajustes_{v}.csv", index=False)


## 3 · Veredictos preregistrados


In [4]:
dec = {v: p3.compare(res, L) for v, res in V.items()}
adopted = [v for v in dec if pr.adopt_phase3(dec[v]["veredicto"])]
print("Adoptadas:", adopted)
pd.concat({v: d["veredicto"] for v, d in dec.items()}, names=["variante"]).reset_index(level=0)


Adoptadas: []


,variante,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,3a1,2,compensa (empate),True,False,"auprc, brier, razon_mae",None,0,0,1,1,2,5,3,3,4,0
1,3a1,3,empate,False,False,"auprc, brier, razon_mae",None,1,0,1,1,2,5,4,3,3,0
2,3a1,4,empate,False,False,"auprc, brier, razon_mae",None,1,0,1,1,4,3,4,3,3,0
0,3a2,2,empate,False,False,NaN,"brier, razon_mae",2,2,3,3,1,0,1,0,0,1
1,3a2,3,empate,False,False,brier,"auprc, razon_mae",1,3,2,4,0,2,3,0,0,1
2,3a2,4,empate,False,False,brier,razon_mae,0,2,2,3,2,1,3,0,0,1
0,3b,2,compensa (empate),False,True,razon_mae,brier,0,1,4,1,4,2,0,3,0,4
1,3b,3,compensa (empate),False,True,"auprc, razon_mae",brier,0,2,3,0,4,3,1,3,0,4
2,3b,4,empate,False,False,"auprc, razon_mae",brier,1,2,3,0,3,3,1,3,0,3


In [5]:
pd.concat({v: d["antelacion"] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte  episodios  diferencia  ic95_inferior  ic95_superior  diferencia_temporada_2010  diferencia_temporada_2015  diferencia_temporada_2016  diferencia_temporada_2022  diferencia_temporada_2023  \
variante                                                                                                                                                                                                            
3a1      0          2        193       0.078          0.041          0.114                      0.067                      0.029                      0.088                      0.105                      0.088   
         1          3        193       0.047          0.010          0.083                     -0.033                      0.000                      0.088                      0.053                      0.088   
         2          4        193       0.047          0.010          0.083                     -0.033                     -0.029                      0.088                      0.053                      0.105   
3a2      0          2        193      -0.026         -0.052          0.000                      0.000                     -0.029                      0.000                      0.000                     -0.070   
         1          3        193      -0.036         -0.067         -0.005                      0.000                      0.000                      0.000                     -0.026                     -0.105   
         2          4        193      -0.031         -0.062          0.000                      0.000                      0.000                      0.000                      0.000                     -0.105   
3b       0          2        193      -0.114         -0.155         -0.073                     -0.033                     -0.059                     -0.206                     -0.079                     -0.158   
         1          3        193      -0.181         -0.233         -0.130                     -0.067                     -0.029                     -0.265                     -0.184                     -0.281   
         2          4        193      -0.171         -0.218         -0.119                     -0.033                     -0.029                     -0.176                     -0.158                     -0.333   

            episodios_temporada_2010  episodios_temporada_2015  episodios_temporada_2016  episodios_temporada_2022  episodios_temporada_2023  
variante                                                                                                                                      
3a1      0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
3a2      0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57  
3b       0                        30                        34                        34                        38                        57  
         1                        30                        34                        34                        38                        57  
         2                        30                        34                        34                        38                        57

In [6]:
pd.concat({v: d["metricas"] for v, d in dec.items()}, names=["variante"]).round(3)


horizonte          bloque  positivos_cand  f1_050_cand  f1_050_ref  recall_050_cand  recall_050_ref  precision_050_cand  precision_050_ref  auprc_cand  auprc_ref  brier_cand  brier_ref  razon_mae_cand  \
variante                                                                                                                                                                                                                
3a1      3           2  temporada_2010           234.0        0.713       0.722            0.671           0.701               0.767              0.745       0.775      0.780       0.031      0.030           0.966   
         8           2  temporada_2015           557.0        0.765       0.772            0.682           0.695               0.870              0.865       0.882      0.885       0.051      0.051           0.952   
         9           2  temporada_2016           395.0        0.652       0.657            0.729           0.714               0.589              0.609       0.741      0.757       0.064      0.059           1.240   
         15          2  temporada_2022           431.0        0.710       0.712            0.724           0.696               0.696              0.728       0.774      0.781       0.055      0.054           1.068   
         16          2  temporada_2023          1039.0        0.858       0.857            0.842           0.828               0.875              0.889       0.946      0.947       0.061      0.062           0.826   
         17          2  temporada_2024           857.0        0.798       0.799            0.853           0.853               0.749              0.752       0.890      0.893       0.075      0.074           1.239   
         21          3  temporada_2010           234.0        0.561       0.631            0.423           0.551               0.832              0.732       0.699      0.706       0.038      0.036           0.939   
         26          3  temporada_2015           557.0        0.572       0.649            0.420           0.515               0.904              0.875       0.837      0.839       0.068      0.064           0.898   
         27          3  temporada_2016           395.0        0.615       0.627            0.646           0.658               0.584              0.599       0.684      0.709       0.071      0.065           1.343   
         33          3  temporada_2022           431.0        0.697       0.704            0.698           0.682               0.695              0.728       0.759      0.771       0.058      0.056           1.125   
         34          3  temporada_2023          1039.0        0.840       0.835            0.812           0.794               0.870              0.880       0.929      0.930       0.071      0.073           0.775   
         35          3  temporada_2024           857.0        0.782       0.788            0.862           0.863               0.715              0.725       0.868      0.876       0.083      0.079           1.375   
         39          4  temporada_2010           234.0        0.135       0.476            0.073           0.359               0.938              0.714       0.594      0.600       0.044      0.042           0.935   
         44          4  temporada_2015           557.0        0.349       0.480            0.214           0.327               0.960              0.901       0.810      0.811       0.081      0.075           0.849   
         45          4  temporada_2016           395.0        0.560       0.590            0.532           0.572               0.585              0.608       0.624      0.660       0.075      0.070           1.391   
         51          4  temporada_2022           431.0        0.639       0.652            0.599           0.599               0.684              0.718       0.728      0.743       0.061      0.060           1.158   
         52          4  temporada_2023          1039.0        0.816       0.811            0.754           

In [7]:
crosses = []
if len(adopted) > 1:
    best = adopted[0]
    for other in adopted[1:]:
        c = p3.compare(V[other], V[best])
        wins = pr.adopt_phase3(c["veredicto"])
        crosses.append({"candidata": other, "frente_a": best, "adopta": wins,
                        "veredicto": "; ".join(f"h{r.horizonte}: {r.veredicto}" for r in c["veredicto"].itertuples())})
        best = other if wins else best
    final = best
else:
    final = adopted[0] if adopted else "L"
print("Base tras 3a1, 3a2 y 3b:", final, crosses)


Base tras 3a1, 3a2 y 3b: L []


## 4 · Lecturas que no deciden


In [8]:
pd.concat({v: d["lectura_sin_2010"] for v, d in dec.items()}, names=["variante"]).reset_index(level=0)


,variante,horizonte,veredicto,mejora_antelacion,empeora_antelacion,vetos,vetos_contra_referencia,a_favor_f1_050,a_favor_auprc,a_favor_brier,a_favor_razon_mae,en_contra_f1_050,en_contra_auprc,en_contra_brier,en_contra_razon_mae,a_favor_antelacion,en_contra_antelacion
0,3a1,2,compensa (empate),True,False,auprc,None,0,0,1,1,2,4,2,2,3,0
1,3a1,3,compensa (empate),True,False,"auprc, brier",None,1,0,1,1,2,4,3,2,3,0
2,3a1,4,compensa (empate),True,False,brier,None,1,0,1,1,3,2,3,2,3,0
0,3a2,2,empate,False,False,None,"brier, razon_mae",2,2,3,3,0,0,1,0,0,1
1,3a2,3,empate,False,False,None,"auprc, razon_mae",1,3,2,3,0,1,2,0,0,1
2,3a2,4,empate,False,False,None,NaN,0,2,2,2,2,1,2,0,0,1
0,3b,2,compensa (empate),False,True,None,brier,0,1,3,1,3,1,0,2,0,4
1,3b,3,pierde,False,True,None,NaN,0,2,2,0,3,2,1,2,0,3
2,3b,4,pierde,False,True,None,NaN,1,2,2,0,3,2,1,2,0,3


In [9]:
rd = {k: p2.readings(v, inp["serie"]) for k, v in {"L": L, **V}.items()}
cols = ["horizonte", "bloque", "positivos", "f1_050", "recall_050", "precision_050", "auprc", "brier",
        "razon_mae", "razon_rmse"]
pd.concat({k: v["metricas_por_temporada"][cols] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque  positivos  f1_050  recall_050  precision_050  auprc  brier  razon_mae  razon_rmse
variante                                                                                                                 
L        3           2  temporada_2010      234.0   0.722       0.701          0.745  0.780  0.030      0.938       0.920
         8           2  temporada_2015      557.0   0.772       0.695          0.865  0.885  0.051      0.945       1.053
         9           2  temporada_2016      395.0   0.657       0.714          0.609  0.757  0.059      1.100       1.071
         10          2  temporada_2017      539.0   0.703       0.744          0.664  0.800  0.068      0.800       0.875
         15          2  temporada_2022      431.0   0.712       0.696          0.728  0.781  0.054      0.962       1.010
         16          2  temporada_2023     1039.0   0.857       0.828          0.889  0.947  0.062      0.847       0.940
         17          2  temporada_2024      857.0   0.799       0.853          0.752  0.893  0.074      1.259       1.566
         21          3  temporada_2010      234.0   0.631       0.551          0.732  0.706  0.036      0.897       0.891
         26          3  temporada_2015      557.0   0.649       0.515          0.875  0.839  0.064      0.895       1.045
         27          3  temporada_2016      395.0   0.627       0.658          0.599  0.709  0.065      1.131       1.142
         28          3  temporada_2017      539.0   0.658       0.664          0.652  0.738  0.076      0.754       0.840
         33          3  temporada_2022      431.0   0.704       0.682          0.728  0.771  0.056      0.959       1.063
         34          3  temporada_2023     1039.0   0.835       0.794          0.880  0.930  0.073      0.791       0.888
         35          3  temporada_2024      857.0   0.788       0.863          0.725  0.876  0.079      1.382       1.778
         39          4  temporada_2010      234.0   0.476       0.359          0.714  0.600  0.042      0.876       0.859
         44          4  temporada_2015      557.0   0.480       0.327          0.901  0.811  0.075      0.847       0.993
         45          4  temporada_2016      395.0   0.590       0.572          0.608  0.660  0.070      1.150       1.176
         46          4  temporada_2017      539.0   0.632       0.590          0.679  0.706  0.081      0.730       0.813
         51          4  temporada_2022      431.0   0.652       0.599          0.718  0.743  0.060      0.952       1.114
         52          4  temporada_2023     1039.0   0.811       0.736          0.901  0.922  0.081      0.748       0.839
         53          4  temporada_2024      857.0   0.761       0.824          0.706  0.840  0.089      1.471       1.892
3a1      3           2  temporada_2010      234.0   0.713       0.671          0.767  0.775  0.031      0.966       0.890
         8           2  temporada_2015      557.0   0.765       0.682          0.870  0.882  0.051      0.952       1.026
         9           2  temporada_2016      395.0   0.652       0.729          0.589  0.741  0.064      1.240       1.146
         10          2  temporada_2017      539.0   0.692       0.766          0.631  0.784  0.073      0.821       0.879
         15          2  temporada_2022      431.0   0.710       0.724          0.696  0.774  0.055      1.068       1.084
         16          2  temporada_2023     1039.0   0.858       0.842          0.875  0.946  0.061      0.826       0.911
         17          2  temporada_2024      857.0   0.798       0.853          0.749  0.890  0.075      1.239       1.423
         21          3  temporada_2010      234.0   0.561       0.423          0.832  0.699  0.038      0.939       0.873
         26          3  temporada_2015      557.0   0.572       0.420          0.904  0.837  0.068      0.898       1.021
         27          3  temporada_2016      395.0   0.615       0.646          0.584  0.684  0.071      1.343

In [10]:
pd.concat({k: v["recall_por_tipo"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque         tipo  semanas  recall  semanas_rango  recall_rango
variante                                                                                         
L        4           2  temporada_2010  brote_nuevo     51.0   0.118              0         0.059
         5           2  temporada_2010     en_curso    183.0   0.863              0         0.033
         14          2  temporada_2015  brote_nuevo     60.0   0.067              0         0.000
         15          2  temporada_2015     en_curso    497.0   0.771              0         0.014
         16          2  temporada_2016  brote_nuevo     53.0   0.132              0         0.000
         17          2  temporada_2016     en_curso    342.0   0.804              0         0.003
         18          2  temporada_2017  brote_nuevo     79.0   0.329              0         0.000
         19          2  temporada_2017     en_curso    460.0   0.815              0         0.004
         26          2  temporada_2022  brote_nuevo     49.0   0.122              0         0.041
         27          2  temporada_2022     en_curso    382.0   0.770              0         0.005
         28          2  temporada_2023  brote_nuevo     86.0   0.174              0         0.000
         29          2  temporada_2023     en_curso    953.0   0.887              0         0.001
         30          2  temporada_2024  brote_nuevo     34.0   0.353              0         0.000
         31          2  temporada_2024     en_curso    823.0   0.874              0         0.004
         36          3  temporada_2010  brote_nuevo     51.0   0.059              0         0.000
         37          3  temporada_2010     en_curso    183.0   0.689              0         0.022
         46          3  temporada_2015  brote_nuevo     60.0   0.033              0         0.000
         47          3  temporada_2015     en_curso    497.0   0.573              0         0.010
         48          3  temporada_2016  brote_nuevo     53.0   0.113              0         0.038
         49          3  temporada_2016     en_curso    342.0   0.743              0         0.003
         50          3  temporada_2017  brote_nuevo     79.0   0.228              0         0.000
         51          3  temporada_2017     en_curso    460.0   0.739              0         0.009
         58          3  temporada_2022  brote_nuevo     49.0   0.163              0         0.000
         59          3  temporada_2022     en_curso    382.0   0.749              0         0.010
         60          3  temporada_2023  brote_nuevo     86.0   0.186              0         0.000
         61          3  temporada_2023     en_curso    953.0   0.849              0         0.006
         62          3  temporada_2024  brote_nuevo     34.0   0.265              0         0.000
         63          3  temporada_2024     en_curso    823.0   0.888              0         0.005
         68          4  temporada_2010  brote_nuevo     51.0   0.020              0         0.000
         69          4  temporada_2010     en_curso    183.0   0.454              0         0.049
         78          4  temporada_2015  brote_nuevo     60.0   0.017              0         0.000
         79          4  temporada_2015     en_curso    497.0   0.364              0         0.012
         80          4  temporada_2016  brote_nuevo     53.0   0.057              0         0.000
         81          4  temporada_2016     en_curso    342.0   0.652              0         0.006
         82          4  temporada_2017  brote_nuevo     79.0   0.203              0         0.013
         83          4  temporada_2017     en_curso    460.0   0.654              0         0.007
         90          4  temporada_2022  brote_nuevo     49.0   0.122              0         0.020
         91          4  temporada_2022     en_curso    382.0   0.660              0         0.024
         92          4  temporada_2023  brote_nuevo     86.0   0.174              0         0

In [11]:
pd.concat({k: v["hu0007_4"] for k, v in rd.items()}, names=["variante"])


horizonte          bloque  precision_050  recall_050    f1_050  razon_mae  razon_rmse  cumple_precision_050  cumple_recall_050  cumple_f1_050  cumple_razon_mae  cumple_razon_rmse  cumple_clasificacion  \
variante                                                                                                                                                                                                                
L        0           2  temporada_2010       0.745455    0.700855  0.722467   0.938085    0.919739                  True              False           True             False              False                 False   
         1           2  temporada_2015       0.865169    0.694794  0.772455   0.945156    1.053429                  True              False           True             False              False                 False   
         2           2  temporada_2016       0.609071    0.713924  0.657343   1.100044    1.071417                  True              False          False             False              False                 False   
         3           2  temporada_2017       0.663921    0.743970  0.703316   0.800489    0.875387                  True              False           True              True              False                 False   
         4           2  temporada_2022       0.728155    0.696056  0.711905   0.961816    1.010115                  True              False           True             False              False                 False   
         5           2  temporada_2023       0.888545    0.827719  0.857428   0.846863    0.940025                  True               True           True              True              False                  True   
         6           2  temporada_2024       0.752058    0.852975  0.799344   1.259388    1.565843                  True               True           True             False              False                  True   
         7           3  temporada_2010       0.731844    0.551282  0.630807   0.896763    0.890730                  True              False          False             False              False                 False   
         8           3  temporada_2015       0.875000    0.515260  0.648588   0.894666    1.044787                  True              False          False             False              False                 False   
         9           3  temporada_2016       0.599078    0.658228  0.627262   1.130717    1.142470                 False              False          False             False              False                 False   
         10          3  temporada_2017       0.652095    0.664193  0.658065   0.754253    0.840480                  True              False          False              True               True                 False   
         11          3  temporada_2022       0.727723    0.682135  0.704327   0.959157    1.063237                  True              False           True             False              False                 False   
         12          3  temporada_2023       0.880342    0.794033  0.835020   0.790613    0.887695                  True              False           True              True              False                 False   
         13          3  temporada_2024       0.725221    0.863477  0.787814   1.382340    1.777912                  True               True           True             False              False                  True   
         14          4  temporada_2010       0.714286    0.358974  0.475921   0.875766    0.858832                  True              False          False             False              False                 False   
         15          4  temporada_2015       0.900990    0.326750  0.479578   0.846978    0.993067                  True              False          False              True              False                 False   
         16          4  temporada_2016       0.608108    0.572152  0.590377   1.150254    1.175559         

In [12]:
pd.concat({k: v["antelacion_semilla_17"] for k, v in rd.items()}, names=["variante"]).round(3)


horizonte          bloque               alerta  episodios  avisados_2_o_mas  ic95_inferior  ic95_superior
variante                                                                                                               
L        2            2  temporada_2010           alerta_050         30             0.033          0.000          0.100
         7            2  temporada_2015           alerta_050         34             0.059          0.000          0.147
         8            2  temporada_2016           alerta_050         34             0.206          0.088          0.353
         9            2  temporada_2017           alerta_050         56             0.446          0.321          0.571
         13           2  temporada_2022           alerta_050         38             0.132          0.026          0.237
         14           2  temporada_2023           alerta_050         57             0.193          0.105          0.298
         15           2  temporada_2024           alerta_050         23             0.435          0.217          0.652
         19           3  temporada_2010           alerta_050         30             0.067          0.000          0.167
         24           3  temporada_2015           alerta_050         34             0.029          0.000          0.088
         25           3  temporada_2016           alerta_050         34             0.265          0.118          0.412
         26           3  temporada_2017           alerta_050         56             0.446          0.321          0.571
         30           3  temporada_2022           alerta_050         38             0.184          0.079          0.316
         31           3  temporada_2023           alerta_050         57             0.316          0.211          0.439
         32           3  temporada_2024           alerta_050         23             0.565          0.348          0.783
         36           4  temporada_2010           alerta_050         30             0.033          0.000          0.100
         41           4  temporada_2015           alerta_050         34             0.029          0.000          0.088
         42           4  temporada_2016           alerta_050         34             0.176          0.059          0.324
         43           4  temporada_2017           alerta_050         56             0.411          0.286          0.536
         47           4  temporada_2022           alerta_050         38             0.158          0.053          0.289
         48           4  temporada_2023           alerta_050         57             0.368          0.246          0.491
         49           4  temporada_2024           alerta_050         23             0.522          0.304          0.739
         53           2  temporada_2010  alerta_persistencia         30             0.033          0.000          0.100
         58           2  temporada_2015  alerta_persistencia         34             0.059          0.000          0.147
         59           2  temporada_2016  alerta_persistencia         34             0.118          0.029          0.235
         60           2  temporada_2017  alerta_persistencia         56             0.161          0.071          0.250
         64           2  temporada_2022  alerta_persistencia         38             0.105          0.026          0.211
         65           2  temporada_2023  alerta_persistencia         57             0.123          0.053          0.211
         66           2  temporada_2024  alerta_persistencia         23             0.130          0.000          0.261
         70           3  temporada_2010  alerta_persistencia         30             0.067          0.000          0.167
         75           3  temporada_2015  alerta_persistencia         34             0.059          0.000          0.147
         76           3  temporada_2016  alerta_persistencia         34             0.176          0.059          0.324
         77           3  temporada_2017  alerta_persis

In [13]:
# 3a2: pesos efectivos no se guardan por fila; 3b: filas y respaldos por régimen (semilla 17)
fits_3b = V["3b"]["ajustes"]
fits_3b[fits_3b.semilla.eq(pr.SEEDS[0])].drop(columns=["corte"]).round(3)


,bloque,regimen,filas_entrenamiento,respaldo,alpha_h2,alpha_h3,alpha_h4,semilla
0,temporada_2008,0,8589,False,4.509,6.586,7.578,17
1,temporada_2008,1,8645,True,2.944,4.637,6.014,17
2,temporada_2009,0,11895,False,7.688,9.186,11.074,17
3,temporada_2009,1,130,False,0.374,0.430,0.448,17
4,temporada_2010,0,15152,False,7.493,9.782,12.685,17
5,temporada_2010,1,318,False,0.917,1.197,1.526,17
6,temporada_2011,0,18308,False,6.795,8.787,11.414,17
7,temporada_2011,1,542,False,0.831,1.178,1.497,17
8,temporada_2012,0,21662,False,7.349,9.807,12.937,17
9,temporada_2012,1,568,False,0.839,1.200,1.555,17


In [14]:
report = [pr.block_name(s) for s in (*pr.DECIDING_SEASONS, pr.CONTRAST_SEASON)]
reg = {k: p3.by_regime(v, inp["serie"], report) for k, v in (("L", L), ("3b", V["3b"]))}
pd.concat({k: v["brier_por_regimen"] for k, v in reg.items()}, names=["variante"]).round(4)


horizonte          bloque  regimen       n  positivos   brier  prob_media  prevalencia  n_rango  positivos_rango  brier_rango  prob_media_rango  prevalencia_rango
variante                                                                                                                                                                       
L        0           2  temporada_2010        0  3147.0       73.0  0.0198      0.0342       0.0232        0                0       0.0001            0.0008                0.0
         1           2  temporada_2010        1   233.0      161.0  0.1719      0.6356       0.6910        0                0       0.0006            0.0042                0.0
         2           2  temporada_2015        0  2902.0      133.0  0.0319      0.0361       0.0458        0                0       0.0001            0.0005                0.0
         3           2  temporada_2015        1   543.0      424.0  0.1531      0.6274       0.7808        0                0       0.0004            0.0020                0.0
         4           2  temporada_2016        0  2972.0      138.0  0.0430      0.0798       0.0464        0                0       0.0003            0.0011                0.0
         5           2  temporada_2016        1   408.0      257.0  0.1783      0.7136       0.6299        0                0       0.0009            0.0029                0.0
         6           2  temporada_2022        0  2940.0      141.0  0.0371      0.0569       0.0480        0                0       0.0002            0.0009                0.0
         7           2  temporada_2022        1   440.0      290.0  0.1650      0.6747       0.6591        0                0       0.0005            0.0025                0.0
         8           2  temporada_2023        0  2405.0      175.0  0.0532      0.0686       0.0728        0                0       0.0003            0.0010                0.0
         9           2  temporada_2023        1   975.0      864.0  0.0831      0.8027       0.8862        0                0       0.0004            0.0010                0.0
         10          2  temporada_2024        0  2455.0      167.0  0.0503      0.0873       0.0680        0                0       0.0003            0.0010                0.0
         11          2  temporada_2024        1   925.0      690.0  0.1352      0.7746       0.7459        0                0       0.0007            0.0027                0.0
         12          3  temporada_2010        0  3147.0       91.0  0.0241      0.0416       0.0289        0                0       0.0002            0.0012                0.0
         13          3  temporada_2010        1   233.0      143.0  0.1973      0.5475       0.6137        0                0       0.0006            0.0033                0.0
         14          3  temporada_2015        0  2907.0      161.0  0.0388      0.0355       0.0554        0                0       0.0002            0.0004                0.0
         15          3  temporada_2015        1   538.0      396.0  0.2013      0.5206       0.7361        0                0       0.0006            0.0025                0.0
         16          3  temporada_2016        0  2971.0      160.0  0.0487      0.0911       0.0539        0                0       0.0005            0.0021                0.0
         17          3  temporada_2016        1   409.0      235.0  0.1865      0.6502       0.5746        0                0       0.0019            0.0041                0.0
         18          3  temporada_2022        0  2936.0      140.0  0.0398      0.0612       0.0477        0                0       0.0002            0.0010                0.0
         19          3  temporada_2022        1   444.0      291.0  0.1637      0.6251       0.6554        0                0       0.0009            0.0041                0.0
         20          3  temporada_2023        0  2441.0      223.0  0.0622      0.0773       0.0914        0                0       0.0002   

In [15]:
curves = pd.concat({k: v["fiabilidad_por_regimen_semilla_17"] for k, v in reg.items()}, names=["variante"])
curves.round(3)


bin     n  prob_media  frecuencia_observada  horizonte          bloque  regimen
variante                                                                                            
L        0    [0.0, 0.1)  2861       0.011                 0.009          2  temporada_2010        0
         1    [0.1, 0.2)   130       0.144                 0.085          2  temporada_2010        0
         2    [0.2, 0.3)    62       0.255                 0.129          2  temporada_2010        0
         3    [0.3, 0.4)    47       0.356                 0.106          2  temporada_2010        0
         4    [0.4, 0.5)    33       0.463                 0.424          2  temporada_2010        0
...                  ...   ...         ...                   ...        ...             ...      ...
3b       272  [0.5, 0.6)    84       0.554                 0.381          4  temporada_2024        1
         273  [0.6, 0.7)   120       0.654                 0.658          4  temporada_2024        1
         274  [0.7, 0.8)   161       0.748                 0.640          4  temporada_2024        1
         275  [0.8, 0.9)   237       0.854                 0.827          4  temporada_2024        1
         276  [0.9, 1.0)   253       0.941                 0.925          4  temporada_2024        1

[590 rows x 7 columns]

## 5 · Salidas: métricas y MLflow


In [16]:
to_json = lambda d: {k: diag.to_json(v) for k, v in d.items() if isinstance(v, pd.DataFrame)}
content = {
    "dispositivo": device, "guardas": {"G8": g8},
    "veredictos": {v: to_json(d) for v, d in dec.items()}, "adoptadas": adopted, "cruces": crosses,
    "base_tras_3b": final, "lecturas": {k: to_json(v) for k, v in rd.items()},
    "ajustes_3b": diag.to_json(V["3b"]["ajustes"].drop(columns=["corte"])),
    "por_regimen": {k: to_json(v) for k, v in reg.items()},
}
content = json.loads(json.dumps(content, default=lambda o: o.item() if hasattr(o, "item") else str(o)))
for section, value in content.items():
    diag.save_metrics_section(METRICS, section, value)
tracking.log_study(
    "mejora_modelo_fase3_inicios", pd.DataFrame(columns=["numero", "estado"]),
    parent={"params": {"fase": "3 (3a1, 3a2, 3b)", "preregistro": "6574669", "base_tras_3b": final,
                       **{f"veredicto.{v}": "; ".join(f"h{r.horizonte}: {r.veredicto}"
                                                    for r in d["veredicto"].itertuples()) for v, d in dec.items()},
                       **{f"xgb.{k}": str(x) for k, x in device.items()}},
            "metricas": {}},
    artifacts=[METRICS], tables={f"veredicto_{v}": d["veredicto"] for v, d in dec.items()},
    description="Fase 3 del estudio 06: pesos de inicios, pesos por temporada y dos regímenes sobre L.")


C:\Users\Rosa\RufoEsEternoElRegreso\Denguekay\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


MLflow: estudio f563d64ea4ae4d3da49a385b0ce3cce6 en 'denguekay/mejora_modelo_fase3_inicios' (sqlite:///C:/Users/Rosa/RufoEsEternoElRegreso/Denguekay/mlflow.db)


'f563d64ea4ae4d3da49a385b0ce3cce6'